# 06 — InSAR & Coherence Analysis

Interferometric SAR processing with NISAR data: read GUNW products,
form interferograms from GSLC pairs, estimate coherence, convert
unwrapped phase to displacement, and visualise results on interactive maps.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bullocke/nice-sar/blob/main/notebooks/06_insar_coherence.ipynb)

## 1. Install and Import

In [ ]:
%pip install -q "fsspec<=2025.3.0" "s3fs<=2025.3.0"
%pip install -q --force-reinstall --no-deps "git+https://github.com/bullocke/nice-sar.git@main"
%pip install -q folium

In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt
import h5py

from nice_sar.io.hdf5 import open_nisar, get_frequencies, get_polarizations
from nice_sar.io.products import (
    read_gunw,
    read_gslc,
    read_identification,
)
from nice_sar.analysis.insar import (
    form_interferogram,
    estimate_coherence,
    phase_to_displacement,
    apply_ionospheric_correction,
    mask_by_coherence,
)
from nice_sar.preprocess.multilook import multilook, multilook_complex

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 2. Read a GUNW Product

NISAR GUNW products contain geocoded unwrapped interferograms at 80 m posting.
Each product includes: unwrapped phase, coherence magnitude, wrapped
interferogram, connected components, and an ionospheric phase screen.

In [ ]:
# Replace with a real GUNW file path or S3 URI
gunw_path = "path/to/NISAR_L2_PR_GUNW_*.h5"

# Read the unwrapped phase
phase = read_gunw(gunw_path, polarization="HH", layer="unwrappedPhase")
print(f"Phase shape: {phase.shape}, dtype: {phase.dtype}")
print(f"CRS: {phase.attrs['crs']}")
print(f"Orbit: {phase.attrs['orbit']}, Track: {phase.attrs['track']}")

In [ ]:
# Read coherence magnitude
coherence = read_gunw(gunw_path, polarization="HH", layer="coherenceMagnitude")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(phase.values, cmap="RdBu_r")
axes[0].set_title("Unwrapped Phase (radians)")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(coherence.values, cmap="magma", vmin=0, vmax=1)
axes[1].set_title("Coherence Magnitude")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

## 3. Phase to Displacement

Convert unwrapped phase to line-of-sight displacement using:

$$d_{LOS} = -\frac{\lambda \phi}{4\pi}$$

where $\lambda = 0.24$ m for NISAR L-band.

In [ ]:
# Apply ionospheric correction first
iono = read_gunw(gunw_path, polarization="HH", layer="ionospherePhaseScreen")
phase_corrected = apply_ionospheric_correction(phase.values, iono.values)

# Convert to displacement
displacement = phase_to_displacement(phase_corrected)

# Mask low-coherence pixels
disp_masked = mask_by_coherence(displacement, coherence.values, threshold=0.3)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(disp_masked, cmap="RdBu_r", vmin=-0.05, vmax=0.05)
ax.set_title("LOS Displacement (m) — coherence > 0.3")
plt.colorbar(im, ax=ax, label="Displacement (m)")
plt.show()

## 4. Form Interferogram from GSLC Pair

For custom interferometry (e.g. non-standard temporal baselines), form
interferograms directly from coregistered GSLC products.

In [ ]:
# Replace with real GSLC file paths
gslc_ref_path = "path/to/NISAR_L2_PR_GSLC_ref.h5"
gslc_sec_path = "path/to/NISAR_L2_PR_GSLC_sec.h5"

ref = read_gslc(gslc_ref_path, polarization="HH")
sec = read_gslc(gslc_sec_path, polarization="HH")

# Form interferogram and estimate coherence
result = form_interferogram(ref.values, sec.values, coherence_window=7)

print(f"Interferogram shape: {result.interferogram.shape}")
print(f"Mean coherence: {np.nanmean(result.coherence):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].imshow(np.angle(result.interferogram), cmap="hsv", vmin=-np.pi, vmax=np.pi)
axes[0].set_title("Interferometric Phase")

axes[1].imshow(result.coherence, cmap="magma", vmin=0, vmax=1)
axes[1].set_title("Coherence")

axes[2].imshow(result.amplitude, cmap="gray")
axes[2].set_title("Amplitude")

plt.tight_layout()
plt.show()

## 5. Multilook Interferogram

Apply complex multilooking to reduce phase noise while preserving
interferometric structure.

In [ ]:
# Complex multilook the interferogram
ifg_ml = multilook_complex(result.interferogram, looks_y=3, looks_x=3)

# Power multilook the coherence
coh_ml = multilook(result.coherence, looks_y=3, looks_x=3)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(np.angle(ifg_ml), cmap="hsv", vmin=-np.pi, vmax=np.pi)
axes[0].set_title("Multilooked Phase (3×3)")

axes[1].imshow(coh_ml, cmap="magma", vmin=0, vmax=1)
axes[1].set_title("Multilooked Coherence (3×3)")

plt.tight_layout()
plt.show()

## 6. Interactive Map Overlay

Overlay coherence or displacement on an interactive folium map.

In [ ]:
from nice_sar.viz.mapping import overlay_raster, add_layer_control
from pyproj import CRS
from rasterio.transform import Affine

crs = CRS.from_user_input(coherence.attrs["crs"])
transform = Affine(*coherence.attrs["transform"])

m = overlay_raster(
    coherence.values, crs, transform,
    cmap="magma", name="Coherence", vmin=0, vmax=1,
)
m = overlay_raster(
    disp_masked, crs, transform,
    map_obj=m, cmap="RdBu_r", name="Displacement",
    vmin=-0.05, vmax=0.05,
)
add_layer_control(m)
m

## 7. Export to GeoTIFF

In [ ]:
from nice_sar.io.geotiff import write_geotiff

write_geotiff("displacement_los.tif", disp_masked, crs=crs, transform=transform)
write_geotiff("coherence.tif", coherence.values, crs=crs, transform=transform)
logger.info("Exported displacement and coherence GeoTIFFs")